In [ ]:
# import os

# os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
# os.environ["CUDA_LAUNCH_BLOCKING"] = "0"

## Loading & Processing the dataset

In [ ]:
from datasets import load_dataset, Dataset, DatasetDict, concatenate_datasets, Features, Image, Value
from PIL import Image as PILImage
import random, pandas as pd
from collections import defaultdict

from datasets import load_dataset
ds = load_dataset("SimulaMet-HOST/Kvasir-VQA")["raw"]

In [ ]:
from datasets import concatenate_datasets

# Remove invalid question entries
valid_ds = ds.filter(lambda ex: ex["question"] and ex["question"] != "none")

# Identify abnormal samples
abnormal_ids = set(
    ex["img_id"] for ex in valid_ds if ex["source"].lower() != "normal"
)
seen_ids = set()
added_examples = []
for ex in valid_ds:
    if ex["img_id"] in abnormal_ids and ex["img_id"] not in seen_ids:
        added_examples.append({
            "image": ex["image"],
            "source": ex["source"],
            "question": "Does this image contain any finding?",
            "answer": "yes",
            "img_id": ex["img_id"]
        })
        seen_ids.add(ex["img_id"])

# Combine cleaned data with added questions
modified_ds = Dataset.from_list(added_examples)
cleaned_ds = concatenate_datasets([valid_ds, modified_ds])


In [ ]:
print("cleaned size:", len(cleaned_ds))

cleaned size: 62747


In [ ]:
import random
from datasets import load_dataset, DatasetDict

# all unique img_ids
all_ids = sorted(set(cleaned_ds["img_id"]))

# Shuffle & split those IDs into 80/10/10
random.seed(42)
random.shuffle(all_ids)

n = len(all_ids)
n_train = int(0.8 * n)
n_val   = int(0.1 * n)

train_ids = set(all_ids[:n_train])
val_ids   = set(all_ids[n_train : n_train + n_val])
test_ids  = set(all_ids[n_train + n_val :])


In [ ]:
def filter_by_id(example, id_set):
    return example["img_id"] in id_set


In [ ]:
train_ds = cleaned_ds.filter(lambda ex: filter_by_id(ex, train_ids),
                     batched=False)
val_ds   = cleaned_ds.filter(lambda ex: filter_by_id(ex, val_ids),
                     batched=False)
test_ds  = cleaned_ds.filter(lambda ex: filter_by_id(ex, test_ids),
                     batched=False)

In [ ]:
dataset = DatasetDict({
    "train":      train_ds,
    "validation": val_ds,
    "test":       test_ds,
})

print({k: len(v) for k, v in dataset.items()})

{'train': 50105, 'validation': 6287, 'test': 6355}


In [ ]:
dataset_full = dataset

In [ ]:
data = dataset_full.remove_columns(["source", "img_id"])
print("SPLIT SIZES:", {k: len(v) for k, v in data.items()})

SPLIT SIZES: {'train': 50105, 'validation': 6287, 'test': 6355}


In [ ]:
import torch
from peft import LoraConfig
from transformers import (AutoProcessor, BitsAndBytesConfig,
                          PaliGemmaForConditionalGeneration, PaliGemmaProcessor)

DEVICE = "cuda:0"
USE_LORA = True
USE_8BIT = True

processor = PaliGemmaProcessor.from_pretrained(
    "google/paligemma2-3b-pt-224",
    do_image_splitting=False,
    use_fast=True
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=8,
    lora_dropout=0.1,
    target_modules=["q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],
    use_dora=True,
    init_lora_weights="gaussian"
)

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_has_fp16_weight=False,
)

model = PaliGemmaForConditionalGeneration.from_pretrained(
    "google/paligemma2-3b-pt-224",
    quantization_config=bnb_config,
    device_map={"": "cuda:0"},
)

model.add_adapter(lora_config)
model.enable_adapters()


In [ ]:
class PaligemmaVQACollator:
    def __init__(self, processor, device):
        self.processor = processor
        self.device = device

    def __call__(self, examples):
        prompts  = [f"<image> Answer as a medical specialist. {ex['question']}" for ex in examples]
        suffixes = [ex["answer"] for ex in examples]
        images   = [ex["image"].convert("RGB") for ex in examples]

        batch = self.processor(
            text=prompts,
            images=images,
            suffix=suffixes,
            return_tensors="pt",
            padding="longest"
        )
        batch = {k: v.to(self.device) for k, v in batch.items()}
        return batch

data_collator = PaligemmaVQACollator(processor, DEVICE)

In [ ]:

import torch

_img_proc = processor.image_processor
_MEAN_VEC = torch.tensor(_img_proc.image_mean)
_STD_VEC  = torch.tensor(_img_proc.image_std)

def _reshape_stats(pv):

    c = len(_MEAN_VEC)
    shape = [1] * pv.dim()
    ch_axis = next(i for i, s in enumerate(pv.shape) if s == c)
    shape[ch_axis] = c
    mean = _MEAN_VEC.view(*shape).to(pv.device, pv.dtype)
    std  = _STD_VEC.view(*shape).to(pv.device, pv.dtype)
    return mean, std

def _to_unit(pv):
    mean, std = _reshape_stats(pv)
    return pv * std + mean

def _to_norm(unit):
    mean, std = _reshape_stats(unit)
    return (unit - mean) / std


def pgd_perturb_batch(model, batch, eps=8/255, alpha=2/255, num_iter=7, random_start=True):

    was_training = model.training
    model.eval()

    pv = batch["pixel_values"].detach()
    unit_clean = _to_unit(pv).detach()
    other = {k: v for k, v in batch.items() if k != "pixel_values"}

    if num_iter <= 0:
        if was_training: model.train()
        return pv

    delta = (torch.empty_like(unit_clean).uniform_(-eps, eps)
             if random_start else torch.zeros_like(unit_clean)).to(unit_clean.dtype)

    for _ in range(num_iter):
        delta.requires_grad_(True)
        unit_adv = torch.clamp(unit_clean + delta, 0.0, 1.0)
        out = model(pixel_values=_to_norm(unit_adv), **other)
        grad = torch.autograd.grad(out.loss, delta)[0]
        with torch.no_grad():
            delta = delta + alpha * grad.sign()
            delta = torch.clamp(delta, -eps, eps)
            delta = torch.clamp(unit_clean + delta, 0.0, 1.0) - unit_clean
        delta = delta.detach()

    unit_adv = torch.clamp(unit_clean + delta, 0.0, 1.0).detach()
    if was_training:
        model.train()
    return _to_norm(unit_adv).detach()


def fgsm_perturb_batch(model, batch, eps=8/255):
    return pgd_perturb_batch(model, batch, eps=eps, alpha=eps, num_iter=1, random_start=False)

In [ ]:

import random as _random
import torch
from transformers import Trainer

ADV_ATTACK   = "pgd"     
ADV_EPS      = 8/255
ADV_ALPHA    = 2/255
ADV_NUM_ITER = 7
ADV_MIX      = True     
ADV_PROB     = 1.0

class AdversarialTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):

        if not (model.training and torch.is_grad_enabled()):
            outputs = model(**inputs)
            loss = outputs.loss
            return (loss, outputs) if return_outputs else loss

        do_adv = _random.random() < ADV_PROB
        attack = ADV_ATTACK
        if attack == "mixed":
            attack = _random.choice(["pgd", "fgsm"])

        if not do_adv:
            outputs = model(**inputs)
            loss = outputs.loss
            return (loss, outputs) if return_outputs else loss

        if attack == "fgsm":
            adv_pv = fgsm_perturb_batch(model, inputs, eps=ADV_EPS)
        else:
            adv_pv = pgd_perturb_batch(model, inputs, eps=ADV_EPS,
                                       alpha=ADV_ALPHA, num_iter=ADV_NUM_ITER)

        adv_inputs = dict(inputs)
        adv_inputs["pixel_values"] = adv_pv.to(inputs["pixel_values"].dtype)
        adv_outputs = model(**adv_inputs)
        adv_loss = adv_outputs.loss

        if ADV_MIX:
            clean_outputs = model(**inputs)
            loss = 0.5 * clean_outputs.loss + 0.5 * adv_loss
            outputs = adv_outputs
        else:
            loss = adv_loss
            outputs = adv_outputs

        return (loss, outputs) if return_outputs else loss

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    num_train_epochs=2,
    per_device_train_batch_size=1
    ,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    warmup_steps=50,
    learning_rate=1e-4,
    weight_decay=0.01,
    logging_steps=25,
    output_dir=r"paligemma2_lora_adv_training_pgd_fgsm_checkpoint",
    save_strategy="epoch",
    save_steps=250,
    save_total_limit=1,
    eval_strategy="epoch",
    bf16=True,
    remove_unused_columns=False,
    report_to="none",
    dataloader_pin_memory=False,
)

trainer = AdversarialTrainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=data["train"],
    eval_dataset=data["validation"],
)

print(f"Adversarial training: attack={ADV_ATTACK} eps={ADV_EPS:.4f} "
      f"alpha={ADV_ALPHA:.4f} iters={ADV_NUM_ITER} mix={ADV_MIX} prob={ADV_PROB}")
trainer.train()

In [ ]:
save_dir = "paligemma2_lora_adv_training_pgd_fgsm_final"
trainer.save_model(save_dir)
processor.save_pretrained(save_dir)
print("saved to", save_dir)

# Evaluation

In [ ]:
import evaluate
import torch
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import Levenshtein

def clean_answer(text):
    lines = text.strip().split("\n")
    return lines[-1].strip() if lines else text.strip()

In [ ]:
preds1 = []
refs1 = []

model.eval()

for ex in tqdm(data['test']):
    image = ex["image"].convert("RGB")
    question = ex["question"]
    answer = ex["answer"]

    # Build prompt
    prompt = f"<image> Answer as a medical specialist. {question}"

    # Prepare inputs
    inputs = processor(
        text=prompt,
        images=image,
        return_tensors="pt",
        padding="longest"
    ).to(model.device)

    # Generate output
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=50,
            num_beams=1,
            do_sample=False,  
            use_cache=False 
        )


    pred = processor.batch_decode(output_ids, skip_special_tokens=True)[0]
    pred = clean_answer(pred)

    preds1.append(pred)
    refs1.append(answer if isinstance(answer, list) else [answer])

In [ ]:

refs1_single1 = [r[0] for r in refs1]

# Accuracy
accuracy = sum(p in r for p, r in zip(preds1, refs1)) / len(refs1) * 100

# Load metrics
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")
meteor = evaluate.load("meteor")

bleu_res   = bleu.compute(predictions=preds1, references=[[r] for r in refs1_single1])
rouge_res  = rouge.compute(predictions=preds1, references=refs1_single1)
meteor_res = meteor.compute(predictions=preds1, references=refs1_single1)

# Jaccard Similarity
j_scores = []
for r, p in zip(refs1_single1, preds1):
    set_r, set_p = set(r.split()), set(p.split())
    j_scores.append(len(set_r & set_p) / len(set_r | set_p) if (set_r | set_p) else 0)
jaccard = sum(j_scores) / len(j_scores) * 100

# Cosine Similarity 
vectorizer = TfidfVectorizer().fit(refs1_single1 + preds1)
ref_vecs  = vectorizer.transform(refs1_single1)
pred_vecs = vectorizer.transform(preds1)
cos_sims  = cosine_similarity(ref_vecs, pred_vecs).diagonal()
cosine = cos_sims.mean() * 100

# Levenshtein Similarity
def normalized_levenshtein(s1, s2):
    if not s1 and not s2:
        return 0
    return Levenshtein.distance(s1, s2) / max(len(s1), len(s2))

def similarity_score(a_ij, o_q_i, tau=0.5):
    nl = normalized_levenshtein(a_ij, o_q_i)
    return 1 - nl if nl < tau else 0

def average_levenshtein_similarity(ground_truth, predicted):
    total_score = 0
    for refs1, pred in zip(ground_truth, predicted):
        if not pred:
            continue
        max_score = max(similarity_score(ref, pred) for ref in refs1)
        total_score += max_score
    return total_score / len(ground_truth) * 100

levenshtein_score = average_levenshtein_similarity(refs1, preds1)

results = {
    "accuracy (%)": round(accuracy, 2),
    "bleu": bleu_res,
    "rouge": rouge_res,
    "meteor": meteor_res,
    "jaccard (%)": round(jaccard, 2),
    "cosine (%)": round(cosine, 2),
    "levenshtein (%)": round(levenshtein_score, 2)
}

for k, v in results.items():
    print(f"{k}:\n{v}\n")

[nltk_data] Downloading package wordnet to /home/rifat/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/rifat/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/rifat/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


accuracy (%):
88.34

bleu:
{'bleu': 0.8298244989167856, 'precisions': [0.9247434435575826, 0.8872193145365525, 0.8665059184568172, 0.8681289640591966], 'brevity_penalty': 0.9362336476178738, 'length_ratio': 0.9381829296248185, 'translation_length': 12278, 'reference_length': 13087}

rouge:
{'rouge1': 0.9275616324623028, 'rouge2': 0.17738055508522182, 'rougeL': 0.9265083975164632, 'rougeLsum': 0.9266186442720676}

meteor:
{'meteor': 0.5332825207566292}

jaccard (%):
90.73

cosine (%):
77.81

levenshtein (%):
91.59



In [ ]:
results = {
    "accuracy (%)": round(accuracy, 2),
    "bleu": bleu_res,
    "rouge": rouge_res,
    "meteor": meteor_res,
    "jaccard (%)": round(jaccard, 2),
    "cosine (%)": round(cosine, 2),
    "levenshtein (%)": round(levenshtein_score, 2)
}

for k, v in results.items():
    print(f"{k}:\n{v}\n")

accuracy (%):
88.34

bleu:
{'bleu': 0.8298244989167856, 'precisions': [0.9247434435575826, 0.8872193145365525, 0.8665059184568172, 0.8681289640591966], 'brevity_penalty': 0.9362336476178738, 'length_ratio': 0.9381829296248185, 'translation_length': 12278, 'reference_length': 13087}

rouge:
{'rouge1': 0.9275616324623028, 'rouge2': 0.17738055508522182, 'rougeL': 0.9265083975164632, 'rougeLsum': 0.9266186442720676}

meteor:
{'meteor': 0.5332825207566292}

jaccard (%):
90.73

cosine (%):
77.81

levenshtein (%):
91.59



## adversarial robustness

In [ ]:
synonyms_dict = {
    "type": ["kind", "category"],
    "procedure": ["process", "test"],
    "image": ["picture", "visual"],
    "abnormalities": ["irregularities", "issues"],
    "present": ["visible", "detected"],
    "easy": ["simple", "straightforward"],
    "detect": ["identify", "locate"],
    "polyp": ["lesion", "mass", "growth"],
    "size": ["dimension", "measurement"],
    "instrument": ["tool", "equipment"],
    "removed": ["extracted", "taken out"],
    "where": ["in what part", "in which area", "location of"],
    "how many": ["number of", "count of", "total"]
}

In [ ]:
insert_words = [
    "possibly", "likely", "evidently", "apparently", "visibly",
    "clinically", "endoscopically", "approximately", "typically"
]

In [ ]:
import random

def synonym_replacement(question, synonyms_dict):
    words = question.split()
    new_words = [random.choice(synonyms_dict.get(w.lower(), [w])) for w in words]
    return " ".join(new_words)

def random_insertion(question, insert_words):
    words = question.split()
    if not words: return question
    insert_word = random.choice(insert_words)
    insert_pos = random.randint(0, len(words))
    return " ".join(words[:insert_pos] + [insert_word] + words[insert_pos:])

def random_deletion(question, p=0.2):
    words = question.split()
    if len(words) == 1: return question
    return " ".join([w for w in words if random.random() > p])


In [ ]:
attacked_preds = []

model.eval()

for ex in tqdm(data['test']):
    image = ex["image"].convert("RGB")
    question = ex["question"]
    answer = ex["answer"]

    # Apply only synonym replacement
    attacked_q = synonym_replacement(question, synonyms_dict)

    prompt = f"<image> Answer as a medical specialist. {attacked_q}"


    inputs = processor(
        text=prompt,
        images=image,
        return_tensors="pt",
        padding="longest"
    ).to(model.device)

    # Generate prediction
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            use_cache=False
        )

    pred = processor.batch_decode(output_ids, skip_special_tokens=True)[0]
    pred = clean_answer(pred)

    attacked_preds.append(pred)

100%|██████████| 6355/6355 [2:31:56<00:00,  1.43s/it]  


In [ ]:

acc_at_attack = sum(p in r for p, r in zip(attacked_preds, refs1))/ len(refs1) * 100

asr = sum(p1 != p2 for p1, p2 in zip(preds1,attacked_preds)) / len(preds1)* 100

In [ ]:
print(f"Acc@Attack (%): {acc_at_attack:.3f}")
print(f"Attack Success Rate (ASR %): {asr:.3f}")

Acc@Attack (%): 86.987
Attack Success Rate (ASR %): 2.754


In [ ]:

refs1_single = [r[0] for r in refs1]


adv_accuracy = sum(p in r for p, r in zip(attacked_preds, refs1)) / len(refs1) * 100

# Evaluate metrics
bleu_res_adv   = bleu.compute(predictions=attacked_preds, references=[[r] for r in refs1_single])
rouge_res_adv  = rouge.compute(predictions=attacked_preds, references=refs1_single)
meteor_res_adv = meteor.compute(predictions=attacked_preds, references=refs1_single)

# Jaccard Similarity
j_scores_adv = []
for r, p in zip(refs1_single, attacked_preds):
    set_r, set_p = set(r.split()), set(p.split())
    j_scores_adv.append(len(set_r & set_p) / len(set_r | set_p) if (set_r | set_p) else 0)
jaccard_adv = sum(j_scores_adv) / len(j_scores_adv) * 100

# Cosine Similarity 
vectorizer_adv = TfidfVectorizer().fit(refs1_single + attacked_preds)
ref_vecs_adv  = vectorizer_adv.transform(refs1_single)
pred_vecs_adv = vectorizer_adv.transform(attacked_preds)
cos_sims_adv  = cosine_similarity(ref_vecs_adv, pred_vecs_adv).diagonal()
cosine_adv = cos_sims_adv.mean() * 100

# Levenshtein Similarity
def normalized_levenshtein(s1, s2):
    if not s1 and not s2:
        return 0
    return Levenshtein.distance(s1, s2) / max(len(s1), len(s2))

def similarity_score(a_ij, o_q_i, tau=0.5):
    nl = normalized_levenshtein(a_ij, o_q_i)
    return 1 - nl if nl < tau else 0

def average_levenshtein_similarity(ground_truth, predicted):
    total_score = 0
    for refs, pred in zip(ground_truth, predicted):
        if not pred:
            continue
        max_score = max(similarity_score(ref, pred) for ref in refs)
        total_score += max_score
    return total_score / len(ground_truth) * 100

levenshtein_adv = average_levenshtein_similarity(refs1, attacked_preds)

adv_results = {
    "accuracy (%)": round(adv_accuracy, 2),
    "bleu": bleu_res_adv,
    "rouge": rouge_res_adv,
    "meteor": meteor_res_adv,
    "jaccard (%)": round(jaccard_adv, 2),
    "cosine (%)": round(cosine_adv, 2),
    "levenshtein (%)": round(levenshtein_adv, 2)
}

for k, v in adv_results.items():
    print(f"{k}:\n{v}\n")


accuracy (%):
86.99

bleu:
{'bleu': 0.8170875012030037, 'precisions': [0.9127637044880671, 0.8735224586288416, 0.8529023746701847, 0.8534072900158478], 'brevity_penalty': 0.9361523672594153, 'length_ratio': 0.9381065179185452, 'translation_length': 12277, 'reference_length': 13087}

rouge:
{'rouge1': 0.9146585749065281, 'rouge2': 0.17339594903189648, 'rougeL': 0.9137325950546793, 'rougeLsum': 0.9137246029516488}

meteor:
{'meteor': 0.524929519413528}

jaccard (%):
89.35

cosine (%):
76.55

levenshtein (%):
90.29



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def visualize_text_perturbations(data, preds_clean, preds_perturbed, perturb_fn, num_samples=5):
    indices = np.random.choice(len(data), size=num_samples, replace=False)

    for idx in indices:
        sample = data[int(idx)]
        image = sample["image"]
        question = sample["question"]
        answer = sample["answer"]

        perturbed_question = perturb_fn(question)

        plt.figure(figsize=(10, 5))
        plt.imshow(image)
        plt.axis("off")
        plt.title("Text Perturbation Example", fontsize=12)


        full_text = (
            f"Ground Truth: {answer}\n"
            f"Original Q: {question}\n"
            f"Original A: {preds_clean[int(idx)]}\n"
            f"Perturbed Q: {perturbed_question}\n"
            f"Perturbed A: {preds_perturbed[int(idx)]}"
        )
        plt.figtext(0.5, -0.13, full_text, wrap=True, ha="center", fontsize=9)
        plt.tight_layout()
        plt.show()

In [ ]:
visualize_text_perturbations(
    data=data['test'],
    preds_clean=preds1,
    preds_perturbed=attacked_preds,
    perturb_fn=lambda q: synonym_replacement(q, synonyms_dict),
    num_samples=5
)

### with all 3 types of text pertubation

In [ ]:
attacked_preds = []

model.eval()

for ex in tqdm(data['test']):
    image = ex["image"].convert("RGB")
    question = ex["question"]
    answer = ex["answer"]

    #applying all text perturbations
    attacked_q = synonym_replacement(question, synonyms_dict)
    attacked_q = random_insertion(attacked_q, insert_words)
    attacked_q = random_deletion(attacked_q)


    prompt = f"<image> Answer as a medical specialist. {attacked_q}"


    inputs = processor(
        text=prompt,
        images=image,
        return_tensors="pt",
        padding="longest"
    ).to(model.device)

    # Generate prediction
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            use_cache=False
        )


    pred = processor.batch_decode(output_ids, skip_special_tokens=True)[0]
    pred = clean_answer(pred)

    attacked_preds.append(pred)


In [ ]:

acc_at_attack = sum(p in r for p, r in zip(attacked_preds, refs1)) / len(refs1) * 100


asr = sum(p1 != p2 for p1, p2 in zip(preds1, attacked_preds)) / len(preds1) * 100

print(f"Acc@Attack (%): {acc_at_attack:.2f}")
print(f"Attack Success Rate (ASR %): {asr:.2f}")

Acc@Attack (%): 77.56
Attack Success Rate (ASR %): 14.41


In [ ]:

refs1_single = [r[0] for r in refs1]

adv_accuracy = sum(p in r for p, r in zip(attacked_preds, refs1)) / len(refs1) * 100

# Evaluate metrics
bleu_res_adv   = bleu.compute(predictions=attacked_preds, references=[[r] for r in refs1_single])
rouge_res_adv  = rouge.compute(predictions=attacked_preds, references=refs1_single)
meteor_res_adv = meteor.compute(predictions=attacked_preds, references=refs1_single)

# Jaccard Similarity
j_scores_adv = []
for r, p in zip(refs1_single, attacked_preds):
    set_r, set_p = set(r.split()), set(p.split())
    j_scores_adv.append(len(set_r & set_p) / len(set_r | set_p) if (set_r | set_p) else 0)
jaccard_adv = sum(j_scores_adv) / len(j_scores_adv) * 100

# Cosine Similarity
vectorizer_adv = TfidfVectorizer().fit(refs1_single + attacked_preds)
ref_vecs_adv  = vectorizer_adv.transform(refs1_single)
pred_vecs_adv = vectorizer_adv.transform(attacked_preds)
cos_sims_adv  = cosine_similarity(ref_vecs_adv, pred_vecs_adv).diagonal()
cosine_adv = cos_sims_adv.mean() * 100

# Levenshtein Similarity
def normalized_levenshtein(s1, s2):
    if not s1 and not s2:
        return 0
    return Levenshtein.distance(s1, s2) / max(len(s1), len(s2))

def similarity_score(a_ij, o_q_i, tau=0.5):
    nl = normalized_levenshtein(a_ij, o_q_i)
    return 1 - nl if nl < tau else 0

def average_levenshtein_similarity(ground_truth, predicted):
    total_score = 0
    for refs, pred in zip(ground_truth, predicted):
        if not pred:
            continue
        max_score = max(similarity_score(ref, pred) for ref in refs)
        total_score += max_score
    return total_score / len(ground_truth) * 100

levenshtein_adv = average_levenshtein_similarity(refs1, attacked_preds)

adv_results = {
    "accuracy (%)": round(adv_accuracy, 2),
    "bleu": bleu_res_adv,
    "rouge": rouge_res_adv,
    "meteor": meteor_res_adv,
    "jaccard (%)": round(jaccard_adv, 2),
    "cosine (%)": round(cosine_adv, 2),
    "levenshtein (%)": round(levenshtein_adv, 2)
}

for k, v in adv_results.items():
    print(f"{k}:\n{v}\n")


accuracy (%):
77.56

bleu:
{'bleu': 0.7123561658871949, 'precisions': [0.8339387649195641, 0.8177453428077588, 0.8040265035677879, 0.7959682345754429], 'brevity_penalty': 0.8764307408730749, 'length_ratio': 0.8834721479330634, 'translation_length': 11562, 'reference_length': 13087}

rouge:
{'rouge1': 0.8148472303242756, 'rouge2': 0.15029695935675458, 'rougeL': 0.8142092765600784, 'rougeLsum': 0.8136920768419251}

meteor:
{'meteor': 0.4636213081348528}

jaccard (%):
79.53

cosine (%):
69.81

levenshtein (%):
80.39



## Image Perturbations

In [ ]:
from PIL import Image, ImageFilter, ImageEnhance
import torchvision.transforms as T
import numpy as np
import io

# Gaussian noise
def add_gaussian_noise(img, mean=0, std=10):
    np_img = np.array(img).astype(np.float32)
    noise = np.random.normal(mean, std, np_img.shape)
    noisy_img = np.clip(np_img + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(noisy_img)

# Gaussian blur
def apply_blur(img, radius=2):
    return img.filter(ImageFilter.GaussianBlur(radius))

# Brightness shift
def shift_brightness(img, factor=1.5): 
    enhancer = ImageEnhance.Brightness(img)
    return enhancer.enhance(factor)

# JPEG compression
def jpeg_compress(img, quality=30):
    buffer = io.BytesIO()
    img.save(buffer, format='JPEG', quality=quality)
    return Image.open(buffer)


In [ ]:
image_attacked_preds = []

model.eval()

for ex in tqdm(data['test']):
    image = ex["image"].convert("RGB")  # Ensure RGB
    question = ex["question"]
    answer = ex["answer"]

    # Apply image perturbations
    perturbed_image = add_gaussian_noise(image)
    perturbed_image = apply_blur(perturbed_image)
    perturbed_image = shift_brightness(perturbed_image, factor=0.7)


    prompt = f"<image> Answer as a medical specialist. {question}"

    # Prepare input
    inputs = processor(
        text=prompt,
        images=perturbed_image,
        return_tensors="pt",
        padding="longest"
    ).to(model.device)

    # Generate prediction
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            use_cache=False
        )

 
    pred = processor.batch_decode(output_ids, skip_special_tokens=True)[0]
    pred = clean_answer(pred)
    image_attacked_preds.append(pred)


100%|██████████| 6355/6355 [2:36:43<00:00,  1.48s/it]  


In [ ]:

acc_at_attack_img = sum(p in r for p, r in zip(image_attacked_preds, refs1_single1)) / len(refs1_single1) * 100


asr_img = sum(p1 != p2 for p1, p2 in zip(preds1, image_attacked_preds)) / len(preds1) * 100

print(f"Image Acc@Attack (%): {acc_at_attack_img:.2f}")
print(f"Image ASR (%): {asr_img:.2f}")

Image Acc@Attack (%): 89.11
Image ASR (%): 4.20


In [ ]:

refs1_single = [r[0] for r in refs1]


adv_accuracy = sum(p in r for p, r in zip(image_attacked_preds, refs1)) / len(refs1) * 100

# Metrics
bleu_res_adv   = bleu.compute(predictions=image_attacked_preds, references=[[r] for r in refs1_single])
rouge_res_adv  = rouge.compute(predictions=image_attacked_preds, references=refs1_single)
meteor_res_adv = meteor.compute(predictions=image_attacked_preds, references=refs1_single)

# Jaccard Similarity
j_scores_adv = []
for r, p in zip(refs1_single, image_attacked_preds):
    set_r, set_p = set(r.split()), set(p.split())
    j_scores_adv.append(len(set_r & set_p) / len(set_r | set_p) if (set_r | set_p) else 0)
jaccard_adv = sum(j_scores_adv) / len(j_scores_adv) * 100

# Cosine Similarity 
vectorizer_adv = TfidfVectorizer().fit(refs1_single + image_attacked_preds)
ref_vecs_adv  = vectorizer_adv.transform(refs1_single)
pred_vecs_adv = vectorizer_adv.transform(image_attacked_preds)
cos_sims_adv  = cosine_similarity(ref_vecs_adv, pred_vecs_adv).diagonal()
cosine_adv = cos_sims_adv.mean() * 100

# Levenshtein Similarity
levenshtein_adv = average_levenshtein_similarity(refs1, image_attacked_preds)

adv_results = {
    "accuracy (%)": round(adv_accuracy, 2),
    "bleu": bleu_res_adv,
    "rouge": rouge_res_adv,
    "meteor": meteor_res_adv,
    "jaccard (%)": round(jaccard_adv, 2),
    "cosine (%)": round(cosine_adv, 2),
    "levenshtein (%)": round(levenshtein_adv, 2)
}

for k, v in adv_results.items():
    print(f"{k}:\n{v}\n")


accuracy (%):
87.13

bleu:
{'bleu': 0.825895265383816, 'precisions': [0.9197170354528256, 0.889348500517063, 0.8762886597938144, 0.8814935064935064], 'brevity_penalty': 0.9263537143801112, 'length_ratio': 0.928937113165737, 'translation_length': 12157, 'reference_length': 13087}

rouge:
{'rouge1': 0.914651916569468, 'rouge2': 0.1741888016258589, 'rougeL': 0.9140806408229016, 'rougeLsum': 0.9140519470540426}

meteor:
{'meteor': 0.5234009725455441}

jaccard (%):
89.36

cosine (%):
76.61

levenshtein (%):
90.4



## lowering image quality

In [ ]:
image_attacked_preds = []

model.eval()

for ex in tqdm(data['test']):
    image = ex["image"].convert("RGB")  
    question = ex["question"]
    answer = ex["answer"]

    # Apply image perturbations

    perturbed_image = jpeg_compress(image, quality=25)


    prompt = f"<image> Answer as a medical specialist. {question}"

    # Prepare input
    inputs = processor(
        text=prompt,
        images=perturbed_image,
        return_tensors="pt",
        padding="longest"
    ).to(model.device)

    # Generate prediction
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            use_cache=False
        )


    pred = processor.batch_decode(output_ids, skip_special_tokens=True)[0]
    pred = clean_answer(pred)
    image_attacked_preds.append(pred)


In [ ]:

acc_at_attack_img = sum(p in r for p, r in zip(image_attacked_preds, refs1)) / len(refs1) * 100

# Attack success rate
asr_img = sum(p1 != p2 for p1, p2 in zip(preds1, image_attacked_preds)) / len(preds1) * 100

print(f"Image Acc@Attack (%): {acc_at_attack_img:.2f}")
print(f"Image ASR (%): {asr_img:.2f}")

Image Acc@Attack (%): 87.96
Image ASR (%): 2.27


In [ ]:

refs1_single = [r[0] for r in refs1]


adv_accuracy = sum(p in r for p, r in zip(image_attacked_preds, refs1)) / len(refs1) * 100

# Metrics
bleu_res_adv   = bleu.compute(predictions=image_attacked_preds, references=[[r] for r in refs1_single])
rouge_res_adv  = rouge.compute(predictions=image_attacked_preds, references=refs1_single)
meteor_res_adv = meteor.compute(predictions=image_attacked_preds, references=refs1_single)

# Jaccard Similarity
j_scores_adv = []
for r, p in zip(refs1_single, image_attacked_preds):
    set_r, set_p = set(r.split()), set(p.split())
    j_scores_adv.append(len(set_r & set_p) / len(set_r | set_p) if (set_r | set_p) else 0)
jaccard_adv = sum(j_scores_adv) / len(j_scores_adv) * 100

# Cosine Similarity 
vectorizer_adv = TfidfVectorizer().fit(refs1_single + image_attacked_preds)
ref_vecs_adv  = vectorizer_adv.transform(refs1_single)
pred_vecs_adv = vectorizer_adv.transform(image_attacked_preds)
cos_sims_adv  = cosine_similarity(ref_vecs_adv, pred_vecs_adv).diagonal()
cosine_adv = cos_sims_adv.mean() * 100

# Levenshtein Similarity
levenshtein_adv = average_levenshtein_similarity(refs1, image_attacked_preds)

adv_results = {
    "accuracy (%)": round(adv_accuracy, 2),
    "bleu": bleu_res_adv,
    "rouge": rouge_res_adv,
    "meteor": meteor_res_adv,
    "jaccard (%)": round(jaccard_adv, 2),
    "cosine (%)": round(cosine_adv, 2),
    "levenshtein (%)": round(levenshtein_adv, 2)
}


for k, v in adv_results.items():
    print(f"{k}:\n{v}\n")


accuracy (%):
87.96

bleu:
{'bleu': 0.8275079457728846, 'precisions': [0.9237572680370159, 0.8890027322404371, 0.871311293543377, 0.8731263383297645], 'brevity_penalty': 0.9307741717813116, 'length_ratio': 0.9330633453045006, 'translation_length': 12211, 'reference_length': 13087}

rouge:
{'rouge1': 0.923624996823267, 'rouge2': 0.17538382206867426, 'rougeL': 0.9230942116062116, 'rougeLsum': 0.9230285335566628}

meteor:
{'meteor': 0.529874521374333}

jaccard (%):
90.32

cosine (%):
77.43

levenshtein (%):
91.25



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def visualize_perturbed_samples(data, preds_clean, preds_perturbed, perturb_fn, num_samples=5):
    indices = np.random.choice(len(data), size=num_samples, replace=False)

    for idx in indices:
        sample = data[int(idx)]
        image = sample["image"]
        question = sample["question"]
        answer = sample["answer"]

        perturbed_image = perturb_fn(image)

        fig, axs = plt.subplots(1, 2, figsize=(12, 5))
        fig.suptitle(f"Question: {question}", fontsize=11)

        axs[0].imshow(image)
        axs[0].set_title("Original")
        axs[0].axis("off")
        axs[0].text(0, -10, f"GT: {answer}", fontsize=9, color='green')
        axs[0].text(0, -25, f"Pred: {preds_clean[int(idx)]}", fontsize=9, color='blue')

        axs[1].imshow(perturbed_image)
        axs[1].set_title("Perturbed")
        axs[1].axis("off")
        axs[1].text(0, -10, f"Pred: {preds_perturbed[int(idx)]}", fontsize=9, color='red')

        plt.tight_layout()
        plt.show()


In [ ]:

visualize_perturbed_samples(
    data=data['test'],
    preds_clean=preds1,
    preds_perturbed=image_attacked_preds,
    perturb_fn=add_gaussian_noise,
    num_samples=5
)


## PGD and FGSM

In [ ]:
model.eval()

def clean_answer(text):

    if not text:
        return ""

    marker = "Answer as a medical specialist."
    if marker in text:
        text = text.split(marker)[-1]
    lines = [l.strip() for l in text.strip().split("\n") if l.strip()]
    return lines[-1] if lines else text.strip()

In [ ]:

_img_proc = processor.image_processor
_MEAN_VEC = torch.tensor(_img_proc.image_mean)
_STD_VEC  = torch.tensor(_img_proc.image_std)

def _reshape_stats(pv):

    c = len(_MEAN_VEC)

    shape = [1] * pv.dim()
    ch_axis = next(i for i, s in enumerate(pv.shape) if s == c)
    shape[ch_axis] = c
    mean = _MEAN_VEC.view(*shape).to(pv.device, pv.dtype)
    std  = _STD_VEC.view(*shape).to(pv.device, pv.dtype)
    return mean, std

def _to_unit(pv):

    mean, std = _reshape_stats(pv)
    return pv * std + mean

def _to_norm(unit):

    mean, std = _reshape_stats(unit)
    return (unit - mean) / std

_IMAGE_TOKEN_ID = (
    getattr(processor, "image_token_id", None)
    or getattr(model.config, "image_token_index", None)
    or getattr(model.config, "image_token_id", None)
)
if _IMAGE_TOKEN_ID is None:

    try:
        _IMAGE_TOKEN_ID = processor.tokenizer.convert_tokens_to_ids("<image>")
    except Exception:
        _IMAGE_TOKEN_ID = None
print("image_token_id:", _IMAGE_TOKEN_ID)


In [ ]:


_PROMPT_TMPL = "<image> Answer as a medical specialist. {q}"

def build_supervised_inputs(image, question, answer):

    ans = answer if isinstance(answer, str) else answer[0]
    prompt = _PROMPT_TMPL.format(q=question)
    batch = processor(
        text=prompt,
        images=image.convert("RGB"),
        suffix=ans,
        return_tensors="pt",
        padding="longest",
    )
    return {k: v.to(model.device) for k, v in batch.items()}


def build_generation_inputs(image, question):

    prompt = _PROMPT_TMPL.format(q=question)
    inputs = processor(
        text=prompt,
        images=image.convert("RGB"),
        return_tensors="pt",
        padding="longest",
    )
    return {k: v.to(model.device) for k, v in inputs.items()}


def pgd_on_pixels(inputs, eps=8/255, alpha=2/255, num_iter=10, random_start=True):

    model.eval()
    pv = inputs["pixel_values"].detach()
    unit_clean = _to_unit(pv).detach()
    other = {k: v for k, v in inputs.items() if k != "pixel_values"}

    delta = (torch.empty_like(unit_clean).uniform_(-eps, eps)
             if random_start else torch.zeros_like(unit_clean)).to(unit_clean.dtype)

    for _ in range(num_iter):
        delta.requires_grad_(True)
        unit_adv = torch.clamp(unit_clean + delta, 0.0, 1.0)
        out = model(pixel_values=_to_norm(unit_adv), **other)
        grad = torch.autograd.grad(out.loss, delta)[0]
        with torch.no_grad():
            delta = delta + alpha * grad.sign()             
            delta = torch.clamp(delta, -eps, eps)
            delta = torch.clamp(unit_clean + delta, 0.0, 1.0) - unit_clean
        delta = delta.detach()

    unit_adv = torch.clamp(unit_clean + delta, 0.0, 1.0).detach()
    return _to_norm(unit_adv).detach()


def fgsm_on_pixels(inputs, eps=8/255):
    return pgd_on_pixels(inputs, eps=eps, alpha=eps, num_iter=1, random_start=False)





In [ ]:
@torch.no_grad()
def generate_from_pv(adv_pixel_values, question, image_for_template):
    gen = build_generation_inputs(image_for_template, question)
    gen["pixel_values"] = adv_pixel_values.to(gen["pixel_values"].dtype)
    input_len = gen["input_ids"].shape[-1]
    out_ids = model.generate(
        **gen,
        max_new_tokens=50,
        num_beams=1,
        do_sample=False,
        use_cache=False,
    )

    new_ids = out_ids[:, input_len:]
    return clean_answer(processor.batch_decode(new_ids, skip_special_tokens=True)[0])

def run_attack_eval(attack="pgd", n=None, eps=8/255, alpha=2/255, num_iter=10):
    test = data["test"]
    if n is not None:
        test = test.select(range(min(n, len(test))))

    preds, refs = [], []
    for ex in tqdm(test, desc=f"attack={attack}"):
        image, question, answer = ex["image"], ex["question"], ex["answer"]
        ans_str = answer[0] if isinstance(answer, list) else answer

        if attack == "clean":
            gen = build_generation_inputs(image, question)
            input_len = gen["input_ids"].shape[-1]
            with torch.no_grad():
                out_ids = model.generate(**gen, max_new_tokens=50, num_beams=1,
                                         do_sample=False, use_cache=False)
            new_ids = out_ids[:, input_len:]
            pred = clean_answer(processor.batch_decode(new_ids, skip_special_tokens=True)[0])

        elif attack in ("pgd", "fgsm"):
            inp = build_supervised_inputs(image, question, ans_str)
            adv_pv = (fgsm_on_pixels(inp, eps=eps) if attack == "fgsm"
                      else pgd_on_pixels(inp, eps=eps, alpha=alpha, num_iter=num_iter))
            pred = generate_from_pv(adv_pv, question, image)

        else:
            raise ValueError(attack)

        preds.append(pred)
        refs.append(answer if isinstance(answer, list) else [answer])
    return preds, refs



In [ ]:
import evaluate
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

_bleu, _rouge, _meteor = evaluate.load("bleu"), evaluate.load("rouge"), evaluate.load("meteor")

def compute_metrics(preds, refs):
    refs_single = [r[0] for r in refs]
    accuracy = sum(p in r for p, r in zip(preds, refs)) / len(refs) * 100
    bleu = _bleu.compute(predictions=preds, references=[[r] for r in refs_single])["bleu"]
    rouge = _rouge.compute(predictions=preds, references=refs_single)["rougeL"]
    meteor = _meteor.compute(predictions=preds, references=refs_single)["meteor"]
    j = []
    for r, p in zip(refs_single, preds):
        sr, sp = set(r.split()), set(p.split())
        j.append(len(sr & sp) / len(sr | sp) if (sr | sp) else 0)
    jaccard = sum(j) / len(j) * 100
    vec = TfidfVectorizer().fit(refs_single + preds)
    cosine = cosine_similarity(vec.transform(refs_single), vec.transform(preds)).diagonal().mean() * 100
    return {"accuracy": round(accuracy, 2), "bleu": round(bleu, 4),
            "rougeL": round(rouge, 4), "meteor": round(meteor, 4),
            "jaccard": round(jaccard, 2), "cosine": round(cosine, 2)}


In [ ]:
def attack_success_rate(clean_preds, adv_preds):
    changed = sum(c.strip() != a.strip() for c, a in zip(clean_preds, adv_preds))
    return changed / len(clean_preds) * 100

In [ ]:


def is_correct(pred, refs):

    return pred in refs

def run_all_attacks_full(attacks=("clean", "fgsm", "pgd"),
                         n=None, eps=8/255, alpha=2/255, num_iter=10):

    store = {}
    refs_ref = None
    for atk in attacks:
        preds, refs = run_attack_eval(attack=atk, n=n, eps=eps, alpha=alpha, num_iter=num_iter)
        store[atk] = preds
        refs_ref = refs  
        print(f"[done] {atk}: {len(preds)} preds")
    return store, refs_ref



In [ ]:


preds_store, refs = run_all_attacks_full(attacks=("fgsm", "pgd"), n=None,
                                         eps=8/255, alpha=2/255, num_iter=10)
preds_store["clean"] = preds1
refs = refs1

attack=fgsm: 100%|██████████| 6355/6355 [3:24:26<00:00,  1.93s/it]  


[done] fgsm: 6355 preds


attack=pgd: 100%|██████████| 6355/6355 [11:47:31<00:00,  6.68s/it]  

[done] pgd: 6355 preds


In [ ]:
def attack_success_rate_changed(clean_preds, adv_preds):
    changed = sum(c.strip() != a.strip() for c, a in zip(clean_preds, adv_preds))
    return changed / len(clean_preds) * 100

def attack_success_rate_flip(clean_preds, adv_preds, refs):

    flipped, base = 0, 0
    for c, a, r in zip(clean_preds, adv_preds, refs):
        if is_correct(c, r):              
            base += 1
            if not is_correct(a, r):
                flipped += 1
    return (flipped / base * 100) if base else 0.0

clean_preds = preds_store["clean"]

print(f"{'attack':<12}{'accuracy':>10}{'ASR_changed':>13}{'ASR_flip':>10}")
print("-" * 45)
summary = {}
for atk, preds in preds_store.items():
    m = compute_metrics(preds, refs)
    asr_changed = attack_success_rate_changed(clean_preds, preds) if atk != "clean" else 0.0
    asr_flip    = attack_success_rate_flip(clean_preds, preds, refs) if atk != "clean" else 0.0
    summary[atk] = {**m, "ASR_changed": round(asr_changed, 2), "ASR_flip": round(asr_flip, 2)}
    print(f"{atk:<12}{m['accuracy']:>10}{asr_changed:>13.2f}{asr_flip:>10.2f}")

attack        accuracy  ASR_changed  ASR_flip
---------------------------------------------
fgsm             82.27        11.93      8.02
pgd              81.61        12.48      8.62
clean            88.34         0.00      0.00


In [ ]:
metrics = ["accuracy", "bleu", "rougeL", "meteor", "jaccard", "cosine"]
header = f"{'attack':<12}" + "".join(f"{m:>10}" for m in metrics)
print(header); print("-" * len(header))
for atk in preds_store:
    r = summary[atk]
    print(f"{atk:<12}" + "".join(f"{r[m]:>10}" for m in metrics))

attack        accuracy      bleu    rougeL    meteor   jaccard    cosine
------------------------------------------------------------------------
fgsm             82.27     0.759    0.8623    0.4867     84.16     72.49
pgd              81.61    0.7549    0.8542    0.4822     83.44     71.93
clean            88.34    0.8298    0.9263    0.5333     90.73     77.81
